# Tahap 1 — Pembersihan transaksi POS

Notebook ini menjalankan kontrak Tahap 1: CSV transaksi mentah diubah menjadi
transaksi Paid yang terdeduplikasi, memiliki merek yang valid, dan disimpan sebagai
`data/interim/transactions_clean.parquet` untuk Tahap 2.

In [1]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

%matplotlib inline

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
CFG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
SEED = int(CFG["seed"])
np.random.seed(SEED)

def rp(*parts: str) -> Path:
    """Resolve a repo-relative path (parts are path fragments under the repo root)."""
    return ROOT.joinpath(*parts)

BRAND_COLORS = {"OPPO": "#0072B2", "Realme": "#D55E00", "Samsung": "#009E73",
                "Vivo": "#CC79A7", "Xiaomi": "#E69F00"}
ALGO_COLORS = {"SARIMAX": "#0072B2", "RF": "#D55E00", "LSTM": "#009E73"}
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 4.5),
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titlesize": 12})

Pembersihan mempertahankan hanya transaksi berstatus `Paid`, membuang duplikat
baris penuh, lalu memetakan setiap `product_name` ke satu dari lima merek.

In [2]:
import logging

logger = logging.getLogger(__name__)

_BRAND_KEYS: list[tuple[str, str]] = [
    ("xiaomi", "Xiaomi"),
    ("redmi", "Xiaomi"),
    ("oppo", "OPPO"),
    ("samsung", "Samsung"),
    ("realme", "Realme"),
    ("vivo", "Vivo"),
]


def brand_from_product_name(name: object) -> str:
    """Kembalikan merek dari product_name; 'Other' jika tak dikenali."""
    n = str(name).lower()
    for key, brand in _BRAND_KEYS:
        if key in n:
            return brand
    return "Other"


def clean_transactions(raw_path: str | Path, out_path: str | Path | None = None) -> pd.DataFrame:
    """Bersihkan CSV POS mentah menjadi DataFrame transaksi Paid, terdedup, ber-merek.

    Statistik ringkas tersimpan di ``df.attrs['stats']`` untuk assertion DoD.
    """
    raw_path = Path(raw_path)
    raw = pd.read_csv(raw_path, encoding="utf-8-sig")
    n_raw = len(raw)

    raw["created_at"] = pd.to_datetime(raw["created_at"])
    raw["paid_at"] = pd.to_datetime(raw["paid_at"], errors="coerce")

    n_void = int((raw["status"] == "Void").sum())
    paid = raw[raw["status"] == "Paid"].copy()

    n_before = len(paid)
    clean = paid.drop_duplicates().reset_index(drop=True)
    n_dup = n_before - len(clean)

    clean["brand"] = clean["product_name"].map(brand_from_product_name)
    n_other = int((clean["brand"] == "Other").sum())
    assert n_other == 0, (
        f"{n_other} baris ter-map ke 'Other'; product_name tak dikenal BRAND_MAP: "
        f"{list(clean.loc[clean['brand'] == 'Other', 'product_name'].unique()[:5])}"
    )

    clean["week_start"] = clean["created_at"].dt.to_period("W").dt.start_time

    stats = {
        "n_raw": n_raw,
        "n_void": n_void,
        "n_duplicates": n_dup,
        "n_clean": len(clean),
        "units": int(clean["qty"].sum()),
    }
    clean.attrs["stats"] = stats
    logger.info(
        "clean: raw=%d void=%d dup=%d -> clean=%d (Σqty=%d)",
        n_raw, n_void, n_dup, len(clean), stats["units"],
    )

    if out_path is not None:
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        clean.to_parquet(out_path, index=False)
        logger.info("Tersimpan: %s", out_path)

    return clean

Jalankan transformasi dengan input dan output yang selalu di-resolve dari akar
repositori; output ini menjadi kontrak data untuk notebook Tahap 2.

In [3]:
clean = clean_transactions(
    rp("data", "raw", "pos_transactions_raw.csv"),
    rp("data", "interim", "transactions_clean.parquet"),
)
print("Statistik pembersihan:", clean.attrs["stats"])
clean.head()

Statistik pembersihan: {'n_raw': 6320, 'n_void': 76, 'n_duplicates': 19, 'n_clean': 6225, 'units': 6907}


,sales_no,created_at,paid_at,status,order_type,customer_name,product_name,variant_name,sku,category,...,service_charge,rounding,grand_total,payment_method,staff_name,printer_id,note,store,brand,week_start
0,1-1-000040,2022-01-02 10:51:35,2022-01-02 10:51:56.080,Paid,Offline,NaN,OPPO A17,Black,OPP-A17,Handphone,...,0,0,4398000,QRIS,Fajar H.,PRN-02,NaN,Toko_B,OPPO,2021-12-27
1,1-1-000034,2022-01-02 11:29:15,2022-01-02 11:29:49.029,Paid,Offline,NaN,realme C35,Green,RLM-C35,Handphone,...,0,0,2399000,E-Wallet,Dewi R.,PRN-02,NaN,Toko_A,Realme,2021-12-27
2,1-1-000004,2022-01-02 12:06:20,2022-01-02 12:08:33.983,Paid,Offline,NaN,OPPO A16,Silver,OPP-A16,Handphone,...,0,0,2099000,Cicilan 0%,Citra W.,PRN-01,NaN,Toko_B,OPPO,2021-12-27
3,1-1-000031,2022-01-02 14:28:02,2022-01-02 14:29:11.557,Paid,Offline,NaN,Xiaomi Redmi Note 11,Glowing Black,XIA-RN11,Handphone,...,0,0,2799000,Cicilan 0%,Fajar H.,NaN,NaN,Toko_A,Xiaomi,2021-12-27
4,1-1-000056,2022-01-02 16:07:05,2022-01-02 16:07:12.453,Paid,Offline,NaN,Samsung Galaxy A14,Glowing Black,SAM-A14,Handphone,...,0,0,2599000,QRIS,Dewi R.,PRN-02,NaN,Toko_A,Samsung,2021-12-27


In [4]:
st = clean.attrs["stats"]
expected = {"n_raw": 6320, "n_void": 76, "n_duplicates": 19, "n_clean": 6225, "units": 6907}
assert st == expected, f"❌ Tahap 1: {st} != {expected}"
assert clean["brand"].nunique() == 5 and clean["store"].nunique() == 4
assert (clean["brand"] != "Other").all()
print("✅ Tahap 1:", st, "| 5 merek × 4 gerai, tak ada 'Other'")

✅ Tahap 1: {'n_raw': 6320, 'n_void': 76, 'n_duplicates': 19, 'n_clean': 6225, 'units': 6907} | 5 merek × 4 gerai, tak ada 'Other'
